# HCM0421 — GaussianPro 90:10 validation and hyperparameter tuning

Eight paper-faithful configurations are screened at 6K, the top three are retrained at 15K, and the winner is confirmed at 30K. Validation cameras are removed before the GaussianPro camera graph is built.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile
import pandas as pd
from IPython.display import display

REPO_DIR = Path('/kaggle/working/round2_kusanagi')
DATA_ROOT = Path('/kaggle/input/datasets/acomingz/maindataset')
SCENE_ROOT = DATA_ROOT / 'HCM0421'
FILTER_ROOT = Path('/kaggle/working/hcm0421_sharp_train')
TUNING_ROOT = Path('/kaggle/working/hcm0421_tuning_90_10')
GPU = '0'
RUN_FINAL_30K = True
os.environ['CUDA_VISIBLE_DEVICES'] = GPU
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
TUNING_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Extract the exact local source package and build CUDA extensions.
if not (REPO_DIR / 'train.py').exists():
    archives = list(Path('/kaggle/input').rglob('kusanagi-source.zip'))
    if len(archives) != 1:
        raise RuntimeError(f'Expected exactly one kusanagi-source.zip, found: {archives}')
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(REPO_DIR)

try:
    import diff_gaussian_rasterization, simple_knn
except ImportError:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install',
        str(REPO_DIR / 'submodules/diff-gaussian-rasterization'),
        str(REPO_DIR / 'submodules/simple-knn'),
        '--no-build-isolation',
    ], check=True)
subprocess.run(['nvidia-smi'], check=False)


In [ ]:
# Resolve HCM0421 and apply the conservative global-blur filter.
def resolve_train_root(scene_root):
    for root in (scene_root, scene_root / 'train'):
        if (root / 'images').exists() and (root / 'sparse').exists():
            return root
    raise FileNotFoundError(scene_root)

SOURCE = resolve_train_root(SCENE_ROOT)
subprocess.run([
    sys.executable, 'tools/select_sharp_training_images.py',
    '--source', str(SOURCE), '--output', str(FILTER_ROOT),
    '--max-remove-ratio', '0.05', '--tail-quantile', '0.12',
    '--neighbor-window', '3', '--min-neighbor-ratio', '1.45',
], cwd=REPO_DIR, check=True)
blur_summary = json.loads((FILTER_ROOT / 'blur_summary.json').read_text())
display(pd.DataFrame([blur_summary]))
print('Removed images:', blur_summary['removed_names'])


In [ ]:
# Successive halving: 8×6K -> top 3×15K -> winner×30K.
command = [
    sys.executable, 'tools/tune_hcm0421_paper_gaussianpro.py',
    '--source', str(SOURCE),
    '--images', str(FILTER_ROOT / 'images'),
    '--output', str(TUNING_ROOT), '--gpu', GPU,
]
if not RUN_FINAL_30K:
    command.append('--skip-final')
print(' '.join(command))
subprocess.run(command, cwd=REPO_DIR, check=True)


In [ ]:
# Review held-out metrics and the selected configuration.
for path in sorted(TUNING_ROOT.glob('*_leaderboard.csv')):
    print('\n', path.name)
    display(pd.read_csv(path).sort_values('score', ascending=False))
best = json.loads((TUNING_ROOT / 'best_config.json').read_text())
print(json.dumps(best, indent=2))
print('Artifacts:', TUNING_ROOT)
